### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="sepsis_survival_minimal_clinical_records",
    dataset_year="2020",
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C53C8N",
    download_description="""
We download the data from UCI.

wget https://archive.ics.uci.edu/static/public/827/sepsis+survival+minimal+clinical+records.zip && unzip sepsis+survival+minimal+clinical+records.zip && unzip s41598-020-73558-3_sepsis_survival_dataset.zip -d data_files &&  rm sepsis+survival+minimal+clinical+records.zip && rm s41598-020-73558-3_sepsis_survival_dataset.zip
mkdir -p local-data-warehouse/sepsis_survival_minimal_clinical_records && mv data_files local-data-warehouse/sepsis_survival_minimal_clinical_records/
""",
    # References
    academic_reference_bibtex=""""@article{chicco2020survival,
  title={Survival prediction of patients with sepsis from age, sex, and septic episode number alone},
  author={Chicco, Davide and Jurman, Giuseppe},
  journal={Scientific reports},
  volume={10},
  number={1},
  pages={17156},
  year={2020},
  publisher={Nature Publishing Group UK London}
}"
""",
    academic_reference_bibtex_key="hicco2020survival",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
We use only the primary cohort from Norway and ignore the subset and the small data (137 samples) from the South Korea cohort.

- The data has only 3 features. Moreover, any other information (such as timestamps from the collection) are not provided. But this should not limit the applicability of the data, as there should be no temporal leakage.
- We reverse the ordinal encoding of gender and the target variable.
- The data has a lot of naturally occurring duplicates. We keep them in as they make sense they would exist and leave it to the pipeline or methods to handle.
- The alternative subset (the study cohort in the original paper) contains only patients that had a sepsis according to the official Sepsis-3 definition. The paper discusses this in more detail and opts for using/testing all cohorts. We could introduce several datasets from this source but this would bias the benchmark too much towards this specific data distribution. We stick to using all data from the primary cohort.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="sepsis_outcome_after9pt5_days_in_hospital", # we rename the target column during preprocessing
    problem_type="binary_classification",
    objective_metric_name="roc_auc", # paper used various metrics and focused on various thresholds, we stick to ROC AUC for now.
    stratify_on="sepsis_outcome_after9pt5_days_in_hospital",
)

## Preprocessing

In [2]:
import pandas as pd

df = pd.read_csv(dataset_mold.path / "data_files" / "s41598-020-73558-3_sepsis_survival_primary_cohort.csv")
print("Loaded data shape:", df.shape)

# Reverse the ordinal encoding
df["gender"] = df["sex_0male_1female"].replace({0: "Male", 1: "Female"})
df["sepsis_outcome_after9pt5_days_in_hospital"] = df["hospital_outcome_1alive_0dead"].replace({0: "Dead", 1: "Alive"})
df = df.drop(columns=["sex_0male_1female", "hospital_outcome_1alive_0dead"])

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

Loaded data shape: (110204, 4)


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 110,204
Columns: 4

#### Duplicate Report
Total duplicate rows: 108693 (98.63% of dataset)
Duplicate rows ignoring target: 109229 (99.12% of dataset)
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,age_years,episode_number,gender,sepsis_outcome_after9pt5_days_in_hospital
0,47,1,Female,Alive
1,64,1,Female,Alive
2,24,1,Female,Alive
3,77,1,Male,Alive
4,82,1,Female,Alive


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,age_years,int64,0,0.0,101,"82, 84, 81, 83, 80, 86, 79, 85, 87, 68"
1,episode_number,int64,0,0.0,5,"1, 2, 3, 4, 5"
2,gender,object,0,0.0,2,"Male, Female"
3,sepsis_outcome_after9pt5_days_in_hospital,object,0,0.0,2,"Alive, Dead"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
age_years,110204.0,62.735255,24.126806,0.0,100.0
episode_number,110204.0,1.349379,0.751799,1.0,5.0


In [7]:
# Categorical Feature Statistics
cat_stats

value   count    pct
column                                    rank                       
gender                                    1       Male   57973  52.61
                                          2     Female   52231  47.39
sepsis_outcome_after9pt5_days_in_hospital 1      Alive  102099  92.65
                                          2       Dead    8105   7.35

In [8]:
# Target Distribution
target_df

,count,pct
sepsis_outcome_after9pt5_days_in_hospital,,
Alive,102099,92.65
Dead,8105,7.35


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=3, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.


## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
019c14c8-158a-7aa7-a3fe-38daa5b5a567
dedd465f2dd08127866fb35fdffd9d1025379978a5005fa1bcd9412bd8fc22db
